In [99]:
import pandas as pd
import numpy as np
import nflreadpy as nfl


In [100]:
def get_weekly_scorers(season, week):
    pbp = nfl.load_pbp(seasons=[season]).to_pandas()

    # Filter for regular season, Week 1
    week_pbp = pbp[(pbp['week'] == week)]

    # Keep only touchdown plays
    week_tds = week_pbp[week_pbp['touchdown'] == 1]

    # Count TDs per scorer. Prefer id+name if both available, else fall back to name only
    use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week_tds.columns]
    if use_cols:
        scorers = (
            week_tds.dropna(subset=use_cols)
            .groupby(use_cols)
            .size()
            .reset_index(name='tds')
        )
        if 'td_player_id' in use_cols:
            scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
        else:
            scorers = scorers.rename(columns={'td_player_name': 'player'})
    else:
        # Fallback if td_* columns not present; derive from rusher/receiver
        rush = week_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
        rec = week_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
        rush.columns = ['player_id', 'player']
        rec.columns = ['player_id', 'player']
        both = pd.concat([rush, rec], ignore_index=True)
        scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

    return scorers

In [101]:
def get_weekly_results(season,week, model_type):       
    scorers = get_weekly_scorers(2025, week)
    if model_type == 'avg' or model_type == 'intersection':
        predictions = pd.read_csv(f'data/predictions_week_{week}_rf.csv')
    else: 
        predictions = pd.read_csv(f'data/predictions_week_{week}_{model_type}.csv')

    #Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
    predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]

    #Sort by predicted_touchdown_probability in descending order
    predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

    # Join predictions with scorers: prefer player_id, else fall back to name
    if 'player_id' in scorers.columns:
        pred_scored = predictions.merge(
            scorers[['player_id', 'tds']], on='player_id', how='inner'
        )
    else:
        pred_scored = predictions.merge(
            scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
        )

    return pred_scored


In [102]:
def simulate_betting(df, scorers, use_kelly=False):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True), summary


In [122]:
def ev_betting(season, week, model_type):
    scorers = get_weekly_results(season, week, model_type)
    if model_type == 'avg':
        rf_predictions = pd.read_csv(f'data/predictions_week_{week}_rf.csv')
        lgbm_predictions = pd.read_csv(f'data/predictions_week_{week}_lgbm.csv')
        #xgb_predictions = pd.read_csv(f'data/predictions_week_{week}_xgb.csv')
        predictions = pd.concat([rf_predictions, lgbm_predictions])
        # group by relevant columns and take the mean of the predicted_touchdown_probability
        predictions = predictions.groupby(['player_id', 'player_display_name', 'position', 'team', 'price', 'market_implied_prob'])['predicted_touchdown_probability'].mean().reset_index()
        # merge with scorers
        predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)
        predictions['model_edge'] = predictions['predicted_touchdown_probability'] - predictions['market_implied_prob']
        predictions.drop_duplicates(subset=['player_id'], inplace=True)
    
    elif model_type == 'intersection':
        rf_predictions = pd.read_csv(f'data/predictions_week_{week}_rf.csv')
        lgbm_predictions = pd.read_csv(f'data/predictions_week_{week}_lgbm.csv')
        top_rb_rf = rf_predictions[rf_predictions['position'] == 'RB'].sort_values(by='predicted_touchdown_probability', ascending=False).head(5)
        top_wr_rf = rf_predictions[rf_predictions['position'] == 'WR'].sort_values(by='predicted_touchdown_probability', ascending=False).head(10)
        top_wr_rf = top_wr_rf[top_wr_rf['model_edge'] >= 0.00]

        top_rb_lgbm = lgbm_predictions[lgbm_predictions['position'] == 'RB'].sort_values(by='predicted_touchdown_probability', ascending=False).head(5).drop(columns=['model_edge', 'predicted_touchdown_probability'])
        top_wr_lgbm = lgbm_predictions[lgbm_predictions['position'] == 'WR'].sort_values(by='predicted_touchdown_probability', ascending=False).head(10)
        top_wr_lgbm = top_wr_lgbm[top_wr_lgbm['model_edge'] >= 0.00].drop(columns=['model_edge', 'predicted_touchdown_probability'])

        ### take intersection of top_rb_rf and top_rb_lgbm
        top_rb = pd.merge(top_rb_rf, top_rb_lgbm, on=['player_id', 'player_display_name', 'position', 'team', 'price', 'market_implied_prob'], how='inner')
        top_wr = pd.merge(top_wr_rf, top_wr_lgbm, on=['player_id', 'player_display_name', 'position', 'team', 'price', 'market_implied_prob'], how='inner')
        
        ev_rf = rf_predictions[round(rf_predictions['model_edge'], 2) >= 0.05]
        ev_rf = ev_rf.sort_values(by='model_edge', ascending=False).head(5)
        ev_lgbm = lgbm_predictions[round(lgbm_predictions['model_edge'], 2) >= 0.05]
        ev_lgbm = ev_lgbm.sort_values(by='model_edge', ascending=False).head(5)
        
        ev_rf = ev_rf[ev_rf['price'] <= 400]
        ev_lgbm = ev_lgbm[ev_lgbm['price'] <= 400].drop(columns=['model_edge', 'predicted_touchdown_probability'])

        ev = pd.merge(ev_rf, ev_lgbm, on=['player_id', 'player_display_name', 'position', 'team', 'price', 'market_implied_prob'], how='inner')
        
        
        return simulate_betting(ev, scorers)
        
        
    else:
        predictions = pd.read_csv(f'data/predictions_week_{week}_{model_type}.csv')
        predictions.drop_duplicates(subset=['player_id'], inplace=True)



    ev = predictions[round(predictions['model_edge'], 2) >= 0.05] 
    ev = ev[ev['price'] <= 400]
    ev = ev.sort_values(by='model_edge', ascending=False).head(5)

    top_rb = predictions[predictions['position'] == 'RB'].sort_values(by='predicted_touchdown_probability', ascending=False).head(5)
    top_qb = predictions[predictions['position'] == 'QB'].sort_values(by='predicted_touchdown_probability', ascending=False).head(5)
    top_te = predictions[predictions['position'] == 'TE'].sort_values(by='predicted_touchdown_probability', ascending=False).head(5)
    top_wr = predictions[predictions['position'] == 'WR'].sort_values(by='predicted_touchdown_probability', ascending=False).head(5)
    #top_wr = top_wr[top_wr['model_edge'] >= 0.00]
    #top_rb = top_rb[top_rb['model_edge'] >= 0.00]
 
    plays = pd.concat([top_rb, top_wr, ev]).drop_duplicates(subset=['player_id'])
    

    #top = predictions[predictions['predicted_touchdown_probability'] >= 0.60]
    
    return simulate_betting(ev, scorers)


In [123]:
def get_total_results(model_type):
    total_bets = 0
    total_hits = 0
    total_profit = 0
    for i in range(1, 7):
        df, roi = ev_betting(2025, i, model_type)
        #print(df)
        total_bets += roi['bets']
        total_hits += roi['hits']
        total_profit += roi['total_profit']

    print(f"Total bets: {total_bets}")
    print(f"Total hits: {total_hits}")
    print(f"Total profit: {total_profit}")
    print(f"Hit rate: {total_hits / total_bets}")
    print(f"ROI: {total_profit / (total_bets * 10.0)}")

In [124]:
get_total_results("rf")

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 44.0, 'roi': 0.88}

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 56.5, 'roi': 1.13}

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 14.5, 'roi': 0.29}

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': -0.5, 'roi': -0.01}

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 35.0, 'roi': 0.7}

{'bets': 2, 'hits': 0, 'hit_rate': 0.0, 'total_profit': -20.0, 'roi': -1.0}

Total bets: 27
Total hits: 12
Total profit: 129.5
Hit rate: 0.4444444444444444
ROI: 0.47962962962962963


In [125]:
get_total_results("lgbm")

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 23.0, 'roi': 0.46}

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 34.5, 'roi': 0.69}

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 4.5, 'roi': 0.09}

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 14.7, 'roi': 0.294}

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 39.0, 'roi': 0.78}

{'bets': 5, 'hits': 0, 'hit_rate': 0.0, 'total_profit': -50.0, 'roi': -1.0}

Total bets: 30
Total hits: 13
Total profit: 65.7
Hit rate: 0.43333333333333335
ROI: 0.219


In [126]:
get_total_results("xgb")

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 23.0, 'roi': 0.46}

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 14.5, 'roi': 0.29}

{'bets': 5, 'hits': 1, 'hit_rate': 0.2, 'total_profit': -18.5, 'roi': -0.37}

{'bets': 4, 'hits': 2, 'hit_rate': 0.5, 'total_profit': 6.0, 'roi': 0.15}

{'bets': 4, 'hits': 2, 'hit_rate': 0.5, 'total_profit': 25.0, 'roi': 0.625}

{'bets': 4, 'hits': 0, 'hit_rate': 0.0, 'total_profit': -40.0, 'roi': -1.0}

Total bets: 27
Total hits: 9
Total profit: 10.0
Hit rate: 0.3333333333333333
ROI: 0.037037037037037035


In [127]:
get_total_results("avg")

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 23.0, 'roi': 0.46}

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 5.0, 'roi': 0.1}

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': 14.5, 'roi': 0.29}

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': -4.0, 'roi': -0.08}

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 35.0, 'roi': 0.7}

{'bets': 4, 'hits': 0, 'hit_rate': 0.0, 'total_profit': -40.0, 'roi': -1.0}

Total bets: 29
Total hits: 11
Total profit: 33.5
Hit rate: 0.3793103448275862
ROI: 0.11551724137931034


In [128]:
#get_total_results("intersection")

For RB: 
- LGBM top 5 RB is the most effective betting strategy observed. 
- Total bets: 25
- Total hits: 19
- Total profit: 75.35
- Hit rate: 0.76
- ROI: 0.3014

For WR: 
- RF top 10 WR with positive edge is the most effective betting strategy observed. 
- Total bets: 25
- Total hits: 16
- Total profit: 152.5
- Hit rate: 0.64
- ROI: 0.61


For EV: 
- Top 5(Edge) RF predictions >= 5% edge & <= +400 odds: 59% ROI
- RF predictions >= 5% edge & <= +400 odds: 48% ROI



In [129]:
current_week = 7
lgbm_predictions = pd.read_csv(f'data/predictions_week_{current_week}_lgbm.csv')
rf_predictions = pd.read_csv(f'data/predictions_week_{current_week}_rf.csv')
xgb_predictions = pd.read_csv(f'data/predictions_week_{current_week}_xgb.csv')

lgbm_predictions = lgbm_predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]
lgbm_predictions = lgbm_predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

rf_predictions = rf_predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]
rf_predictions = rf_predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

xgb_predictions = xgb_predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]
xgb_predictions = xgb_predictions.sort_values(by='predicted_touchdown_probability', ascending=False)


## Running Back Plays for Current Week

In [130]:
rb_plays = lgbm_predictions[lgbm_predictions['position'] == 'RB'].head(5)
rb_plays

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0039139,Jahmyr Gibbs,RB,DET,0.612123,-210.0,-0.065296,0.677419
1,00-0036223,Jonathan Taylor,RB,IND,0.601900,-195.0,-0.059117,0.661017
3,00-0036997,Javonte Williams,RB,DAL,0.553265,-150.0,-0.046735,0.600000
4,00-0034844,Saquon Barkley,RB,PHI,0.543621,-130.0,-0.021597,0.565217
6,00-0036875,Rhamondre Stevenson,RB,NE,0.512344,100.0,0.012344,0.500000


## Wide Receiver Plays for Current Week

In [131]:
wr_plays = rf_predictions[rf_predictions['position'] == 'WR'].head(5)
#wr_plays = wr_plays[wr_plays['model_edge'] >= 0.00]
wr_plays

,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
8,00-0036963,Amon-Ra St. Brown,WR,DET,0.464654,-120.0,-0.080801,0.545455
9,00-0036358,CeeDee Lamb,WR,DAL,0.463602,115.0,-0.001514,0.465116
10,00-0037247,George Pickens,WR,DAL,0.462987,125.0,0.018542,0.444444
14,00-0038543,Jaxon Smith-Njigba,WR,SEA,0.427406,130.0,-0.007376,0.434783
18,00-0031381,Davante Adams,WR,LA,0.403068,-115.0,-0.131815,0.534884


In [132]:
ev_plays = rf_predictions[round(rf_predictions['model_edge'], 2) >= 0.05].sort_values(by='model_edge', ascending=False) 
ev_plays = ev_plays[ev_plays['price'] <= 400]
ev_plays



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
20,00-0033873,Patrick Mahomes,QB,KC,0.396442,310.0,0.152540,0.243902
33,00-0034351,Dallas Goedert,TE,PHI,0.335260,270.0,0.064989,0.270270
63,00-0038117,Wan'Dale Robinson,WR,NYG,0.267804,350.0,0.045582,0.222222


In [133]:
ev_plays_lgbm = lgbm_predictions[round(lgbm_predictions['model_edge'], 2) >= 0.05].sort_values(by='model_edge', ascending=False) 
ev_plays_lgbm = ev_plays_lgbm[ev_plays_lgbm['price'] <= 400]
ev_plays_lgbm



,player_id,player_display_name,position,team,predicted_touchdown_probability,price,model_edge,market_implied_prob
12,00-0033873,Patrick Mahomes,QB,KC,0.461736,310.0,0.217833,0.243902
2,00-0037247,George Pickens,WR,DAL,0.569052,125.0,0.124608,0.444444
23,00-0037197,Isiah Pacheco,RB,KC,0.385936,215.0,0.068475,0.317460
13,00-0039065,Sam LaPorta,TE,DET,0.448226,150.0,0.048226,0.400000
14,00-0036555,Chuba Hubbard,RB,CAR,0.446842,150.0,0.046842,0.400000


In [134]:
#import data_collection as data

#nfl_teams = pd.read_csv('nfl_teams.csv')
#team_map = dict(zip(nfl_teams['team_name'], nfl_teams['team_id']))

#nfl_data = data.get_all_historic_data([2020, 2021, 2022, 2023, 2024, 2025], team_map)





In [135]:
#nfl_data.to_csv('raw_nfl_data.csv', index=False)